# Section 1

#### Import libraries

In [ ]:
import requests
import bs4
import lxml.html as lh
import pandas as pd
pd.set_option('display.max_rows',300)
import folium

In [ ]:
res = requests.get("https://en.wikipedia.org/wiki/List_of_postal_codes_of_Canada:_M")

In [ ]:
#Store the contents of the website under doc
doc = lh.fromstring(res.content)

In [ ]:
#Parse data that are stored between <tr>..</tr> of HTML
tr_elements = doc.xpath('//tr')

In [ ]:
#Check the length of the first 12 rows
[len(T) for T in tr_elements[:12]]

In [ ]:
#Parse Table Header

tr_elements = doc.xpath('//tr')

In [ ]:
#Create empty list
col=[]
i=0

#For each row, store each first element (header) and an empty list
for t in tr_elements[0]:
    i+=1
    name=t.text_content()
    print('%d:"%s"' %(i,name))
    col.append((name,[]))

In [ ]:
print(col)

In [ ]:
#Since out first row is the header, data is stored on the second row onwards

for j in range(1,len(tr_elements)):
    
    #T is our j'th row
    T=tr_elements[j]
    
    #If row is not of size 3, the //tr data is not from our table 
    if len(T)!=3:
        break
    
    #i is the index of our column
    i=0
    
    #Iterate through each element of the row
    for t in T.iterchildren():
        data=t.text_content() 
        #Check if row is empty
        if i>0:
        #Convert any numerical value to integers
            try:
                data=int(data)
            except:
                pass
        #Append the data to the empty list of the i'th column
        col[i][1].append(data)
        #Increment i for the next column
        i+=1

In [ ]:
print(col)

In [ ]:
[len(C) for (title,C) in col]

In [ ]:
Dict={title:column for (title,column) in col}
df=pd.DataFrame(Dict)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

#### Data cleaning

In [ ]:
df.Borough.str.contains("Not assigned")

In [ ]:
df1 = df[~df.Borough.str.contains("Not assigned")]

In [ ]:
df1.shape

In [ ]:
df1

In [ ]:
#trying out remove \n
df1['Neighbourhood\n'].replace('\n','',regex=True)

In [ ]:
df1

In [ ]:
df1 = df1.replace(r'\n',' ', regex=True) 

In [ ]:
df1

In [ ]:
df2 = df1.copy()  #save a copy first

In [ ]:
df2

In [ ]:
df2.columns

In [ ]:
df2.rename(columns= lambda x: x.strip(), inplace=True)

In [ ]:
df2.columns

In [ ]:
df2.head()  #Stripped \n

In [ ]:
df2.shape

In [ ]:
df2.columns

In [ ]:
df3 = df2.groupby(['Postcode','Borough'], as_index=False).agg(','.join)

In [ ]:
df3.head()

In [ ]:
df3   #See and check all rows

In [ ]:
print("The number of rows of the dataframe:", df3.shape)

In [ ]:
#df3.to_csv('cleaned.csv',index=False) #save a csv file

# Part 2

In [ ]:
#Import the geospatial data

geodata = pd.read_csv('Geospatial_Coordinates.csv')

In [ ]:
geodata.head()

In [ ]:
df4 = df3.merge(geodata, how="left", on="Postcode")

### Dataframe for Q2 is below:

In [ ]:
df4

# Part 3

In [ ]:
latitude = 43.636966
longitude = -79.615819

In [ ]:
# create map of Mississauga using latitude and longitude values
map = folium.Map(location=[latitude, longitude], zoom_start=11)

In [ ]:
map

<img src="map1.jpg">

In [ ]:
#Define markers

tooltip = 'Click me!'

folium.Marker([latitude,longitude], popup='<b>Canada Post Gateway Processing Centre</b>', tooltip=tooltip).add_to(map)

In [ ]:
map

<img src="map2.jpg">

#### Note: There is no nearby neighbourhood for Mississauga